In [1]:
import torch
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Union, Optional, Tuple
from sentence_transformers import SentenceTransformer, CrossEncoder

from nlp4bia.linking.retrievers import DenseRetriever
from nlp4bia.linking.rerankers import CrossEncoderReranker


class BECELinker:
    """
    A unified "Bi-Encoder + Cross-Encoder" entity linker.

    1) Uses a DenseRetriever (bi-encoder) to retrieve top-k candidates from a gazetteer.
    2) Uses a CrossEncoderReranker to re-score and sort those top-k candidates.

    Attributes:
        df_gazetteer (pd.DataFrame): The gazetteer DataFrame with "term" and "code" columns.
        retriever (DenseRetriever): Bi-encoder retriever instance.
        reranker (CrossEncoderReranker): Cross-encoder reranker instance.
    """

    def __init__(
        self,
        df_gazetteer: pd.DataFrame,
        biencoder_model_or_path: Union[SentenceTransformer, str],
        crossencoder_model_or_path: Union[CrossEncoder, str],
        normalize_embeddings: bool = True,
        biencoder_batch_size: int = 32,
        reranker_batch_size: int = 32,
        reranker_device: str = "cuda",
        biencoder_device: str = "cuda",
        show_progress_bar: bool = True
    ) -> None:
        """
        Initialize BECELinker with both bi-encoder and cross-encoder components.

        Args:
            df_gazetteer (pd.DataFrame):
                DataFrame containing the gazetteer. Must contain:
                  - "term": textual candidate
                  - "code": unique identifier for each term
            biencoder_model (SentenceTransformer):
                Pretrained SentenceTransformer instance for encoding queries and,
                if needed, encoding gazetteer terms.
            crossencoder_model_path (str):
                Path (or HuggingFace ID) to a pretrained CrossEncoder for reranking.
            normalize_embeddings (bool, optional):
                If True, L2-normalize gazetteer embeddings and query embeddings.
                Defaults to True.
            biencoder_batch_size (int, optional):
                Batch size for any encoding steps in the bi-encoder. Defaults to 32.
            reranker_batch_size (int, optional):
                Batch size for scoring pairs in the CrossEncoderReranker. Defaults to 32.
            retriever_device (str, optional):
                Device to load the bi-encoder model on (e.g. "cuda" or "cpu"). Defaults to "cuda".
            reranker_device (str, optional):
                Device to load the cross-encoder on. Defaults to "cuda".
            show_progress_bar (bool, optional):
                If True, show tqdm progress bars during encoding and scoring. Defaults to True.

        Raises:
            AssertionError: If `df_gazetteer` lacks the required columns "term" or "code".
        """
        assert "term" in df_gazetteer.columns, "`df_gazetteer` must contain a 'term' column"
        assert "code" in df_gazetteer.columns, "`df_gazetteer` must contain a 'code' column"
        self.df_gazetteer = df_gazetteer.reset_index(drop=True).copy()

        # Initialize DenseRetriever (bi-encoder)
        self.retriever = DenseRetriever(
            df_candidates=self.df_gazetteer,
            model_or_path=biencoder_model_or_path,
            normalize=normalize_embeddings,
            vector_db=None,  # will be computed inside DenseRetriever __init__
            vector_db_batch_size=biencoder_batch_size,
            device=biencoder_device
        )

        # Initialize CrossEncoderReranker
        term2code_mapping = self.df_gazetteer.set_index("term")["code"].to_dict()
        self.reranker = CrossEncoderReranker(
            model_or_path=crossencoder_model_or_path,
            device=reranker_device,
            batch_size=reranker_batch_size,
            term2code=term2code_mapping,
            show_progress_bar=show_progress_bar
        )

    def link(
        self,
        mentions: List[str],
        n_candidates: int = 200,
        top_k: int = 50,
        return_documents: bool = True
    ) -> List[Dict[str, Any]]:
        """
        Perform entity linking for each mention by:
          1) Retrieving the top_k nearest neighbors from the gazetteer via bi-encoder.
          2) Re-ranking those k candidates with the cross-encoder.
          3) Returning the final sorted list for each mention.

        Args:
            mentions (List[str]):
                List of N mention strings to link.
            top_k (int, optional):
                Number of nearest neighbors to retrieve before reranking. Defaults to 50.
            return_documents (bool, optional):
                If True, each output dict includes "mention": the original query string.
                If False, omit that field. Defaults to True.

        Returns:
            List[Dict[str, Any]] of length N. Each dict contains:
              - (optional) "mention": str (only if return_documents=True)
              - "codes": List[str], final candidate codes sorted by cross-encoder score
              - "terms": List[str], final candidate terms sorted by score
              - "similarity": List[float], final cross-encoder scores (higher = better)
        """
        # Step 1: Retrieve top_k candidates via DenseRetriever
        raw_topk = self.retriever.retrieve_top_k(
            data=mentions,
            k=n_candidates,
            input_format="text",
            return_documents=return_documents
        )
        # raw_topk: List of dicts, each with keys:
        #   - "mention" (if return_documents=True)
        #   - "codes": List[str]
        #   - "terms": List[str]
        #   - "similarity": List[float]

        # Prepare candidate dicts for reranking: only "terms" & "codes" are needed
        cand_dicts: List[Dict[str, List[str]]] = []
        for entry in raw_topk:
            cand_dicts.append({
                "terms": entry["terms"],
                "codes": entry["codes"]
            })

        # Step 2: Rerank top_k candidates with CrossEncoder
        reranked = self.reranker.rerank(
            mentions=mentions,
            candidates=cand_dicts,
            k=top_k,  # keep all, but we could also pass top_k again if desired
            return_documents=return_documents
        )
        # reranked: List of dicts, each with keys:
        #   - "mention" (if return_documents=True)
        #   - "terms": sorted terms
        #   - "codes": sorted codes
        #   - "similarity": sorted cross-encoder scores

        return reranked

/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from nlp4bia.datasets.benchmark.medprocner import MedprocnerLoader, MedprocnerGazetteer
from sentence_transformers import SentenceTransformer
from nlp4bia.linking import BECELinker

# 1) Load data
df_proc = MedprocnerLoader().df
gaz_proc = MedprocnerGazetteer().df#.iloc[:100]

# biencoder_path = "/gpfs/projects/bsc14/abecerr1/hub/models--ICB-UMA--ClinLinker-KB-GP/snapshots/8f914c58a1cbcff43331eb15b101eaa5e5c6920a"
# biencoder_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/biencoder_medprocner_1_epoch_32_batch_5_parents_stag"
biencoder_path = "BSC-NLP4BIA/Medprocner-BiEncoder"
# ce_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
ce_path = "BSC-NLP4BIA/Medprocner-CE-Reranker"

# # 2) Prepare a SentenceTransformer bi-encoder (already loaded)
# biencoder_model = SentenceTransformer(biencoder_path)
biencoder_model = SentenceTransformer(biencoder_path, device="cuda")
vector_db = biencoder_model.encode(
    gaz_proc["term"].tolist(),
    batch_size=4096,
    show_progress_bar=True,
    convert_to_tensor=True
)


/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 58/58 [00:27<00:00,  2.11it/s]


In [2]:

# 3) Initialize the BECELinker
linker = BECELinker(
    df_gazetteer=gaz_proc,
    biencoder_model_or_path=biencoder_path,
    crossencoder_model_or_path=ce_path,
    biencoder_batch_size=4096,
    reranker_batch_size=4096,
    vector_db=vector_db,
)

# 4) Link a list of mentions
ls_mentions = df_proc["span"].tolist()[:10]
results = linker.link(
    mentions=ls_mentions,
    n_candidates=200,
    top_k=5,
    return_documents=True
)

# 5) Inspect output
for res in results:
    print(f"Mention: {res['mention']}")
    for idx, (term, code, score) in enumerate(zip(res["terms"], res["codes"], res["similarity"]), start=1):
        print(f"  {idx:02d}. {term} ({code}) → {score:.4f}")
    print()


Initializing DenseRetriever...
Using bi-encoder model: BSC-NLP4BIA/Medprocner-BiEncoder
Note: Vector DB will be computed on the fly. Increase `biencoder_batch_size` to accelerate this.
In case of MemoryError, try reducing `biencoder_batch_size` or using a smaller model.
DenseRetriever initialized successfully.
Initializing CrossEncoder Reranker...
Using CrossEncoder model: BSC-NLP4BIA/Medprocner-CE-Reranker


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.01it/s]

Mention: Auscultación pulmonar
  01. auscultación del tracto respiratorio inferior (449264008) → 0.9679
  02. aire respiratorio (11891009) → 0.2314
  03. sin síntomas respiratorios (161962008) → 0.0601
  04. murmullo cardíaco ausente (301131000) → 0.0407
  05. sin dificultad para respirar (161938003) → 0.0311

Mention: exploración neurológica
  01. examen clínico (5880005) → 0.9707
  02. examen neurológico (84728005) → 0.9626
  03. examen físico (5880005) → 0.9498
  04. estudio clínico (110465008) → 0.9403
  05. exploración física (5880005) → 0.8516

Mention: exploración urológica
  01. examen urológico (302778005) → 0.9717
  02. urología (394612005) → 0.9706
  03. examen del aparato genitourinario (268945009) → 0.9394
  04. exploración del aparato genitourinario (363117004) → 0.8607
  05. examen clínico (5880005) → 0.0760

Mention: palpación
  01. palpación (113011001) → 0.9713
  02. sin dolor a la palpación en región abdominal (860640005) → 0.9239
  03. dolor a la palpación ausente (

In [ ]:
# Upload model to hub
# linker.retriever.model.push_to_hub("BSC-NLP4BIA/Medprocner-Biencoder",
#                                   token="",
#                                   private=False)

model.safetensors: 100%|██████████| 504M/504M [00:44<00:00, 11.2MB/s] 


'https://huggingface.co/BSC-NLP4BIA/Medprocner-Biencoder/commit/94d01512c779bdf42b7c1e7a28bbc3cdf746415e'

In [ ]:
# Upload model to hub
# linker.reranker.model.push_to_hub("BSC-NLP4BIA/Medprocner-CE-Reranker",
#                                   token="",
#                                   private=False)

model.safetensors: 100%|██████████| 504M/504M [00:15<00:00, 31.9MB/s]  


'https://huggingface.co/BSC-NLP4BIA/Medprocner-CE-Reranker/commit/b81d321d210317bbd2965991a9ee826ae4dc2892'